# Detecção de fraude em cartões de crédito

Neste notebook eu treino e comparo modelos que detectam fraude em transações reais de cartão (base do TensorFlow/Kaggle, ~284 mil transações).

**O ponto central:** só ~0,17% das transações são fraude. Por isso a **acurácia engana** e toda a avaliação gira em torno da **classe de fraude (1)**:

| Métrica | Pergunta que responde |
|---|---|
| **Recall** | Das fraudes reais, quantas o modelo encontrou? |
| **Precisão** | Das transações que o modelo marcou como fraude, quantas eram mesmo fraude? |
| **F1** | Equilíbrio entre recall e precisão |
| **PR-AUC** (average precision) | Qualidade geral na curva precisão × recall, a mais útil em dados desbalanceados |

**Roteiro:** 1. Explorar → 2. Preparar → 3. Treinar → 4. Avaliar → 5. Ajustar (Pipeline, limiar, GridSearchCV) → 6. Experimento extra (under/oversampling) → 7. Explicar (importância + SHAP).

In [ ]:
# No Google Colab o shap não vem instalado. XGBoost, scikit-learn e pandas já vêm.
!pip install -q shap

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve, roc_curve,
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.dpi"] = 110

## 1. Explorar os dados

A base é lida direto do link, sem baixar o arquivo (ele passa de 100 MB e **não** vai para o repositório).

- `Time`: segundos desde a primeira transação
- `Amount`: valor da transação
- `V1` a `V28`: variáveis anonimizadas por PCA
- `Class`: 1 = fraude, 0 = normal

In [ ]:
URL = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"
df = pd.read_csv(URL)

print("Formato:", df.shape)
df.head()

In [ ]:
print("Valores nulos:", int(df.isna().sum().sum()))
print("Linhas duplicadas:", int(df.duplicated().sum()), "(mantidas, para seguir o enunciado)")

contagem = df["Class"].value_counts()
proporcao = df["Class"].value_counts(normalize=True)

resumo = pd.DataFrame({
    "transacoes": contagem,
    "proporcao_%": (proporcao * 100).round(3),
})
resumo.index = resumo.index.map({0: "normal (0)", 1: "fraude (1)"})
resumo

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

ax[0].bar(["normal (0)", "fraude (1)"], contagem.sort_index().values, color=["#4C78A8", "#E45756"])
ax[0].set_yscale("log")
ax[0].set_title("Transações por classe (escala log)")
ax[0].set_ylabel("quantidade")

ax[1].hist(df["Amount"], bins=60, color="#4C78A8")
ax[1].set_title("Amount (muito assimétrico)")
ax[1].set_xlabel("valor")

ax[2].hist(np.log1p(df["Amount"]), bins=60, color="#72B7B2")
ax[2].set_title("log1p(Amount) (bem mais comportado)")
ax[2].set_xlabel("log(1 + valor)")

plt.tight_layout()
plt.show()

df.groupby("Class")["Amount"].describe().round(2)

## 2. Preparar os dados

1. `Amount` é muito assimétrico, então crio `log_amount = log1p(Amount)` e descarto o `Amount` original.
2. Separo treino e teste **com `stratify=y`**, para que os dois tenham a mesma proporção de fraudes (~0,17%).
3. Padronizo com `StandardScaler` **ajustado só no treino**. Ajustar no dataset inteiro deixaria informação do teste vazar para o treino.

In [ ]:
df_model = df.copy()
df_model["log_amount"] = np.log1p(df_model["Amount"])
df_model = df_model.drop(columns=["Amount"])

X = df_model.drop(columns=["Class"])
y = df_model["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Treino:", X_train.shape, "| fraudes:", int(y_train.sum()), f"({y_train.mean()*100:.3f}%)")
print("Teste: ", X_test.shape,  "| fraudes:", int(y_test.sum()),  f"({y_test.mean()*100:.3f}%)")

In [ ]:
scaler = StandardScaler().fit(X_train)   # aprende média/desvio só com o treino

X_train_s = pd.DataFrame(scaler.transform(X_train), columns=X.columns, index=X_train.index)
X_test_s  = pd.DataFrame(scaler.transform(X_test),  columns=X.columns, index=X_test.index)

X_train_s.describe().loc[["mean", "std"]].round(2).iloc[:, :6]

### Por que a acurácia engana

Antes de treinar qualquer modelo de verdade, um "modelo" que responde **sempre "não é fraude"**:

In [ ]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_train_s, y_train)
pred_dummy = dummy.predict(X_test_s)

print(f"Acurácia : {accuracy_score(y_test, pred_dummy):.4%}")
print(f"Recall   : {recall_score(y_test, pred_dummy):.4%}   <- não pegou nenhuma fraude")
print(f"Precisão : {precision_score(y_test, pred_dummy, zero_division=0):.4%}")
print(f"F1       : {f1_score(y_test, pred_dummy, zero_division=0):.4%}")

Acurácia altíssima e recall zero: o modelo parece ótimo e não serve para nada. A partir daqui, **recall, precisão, F1 e PR-AUC da classe de fraude** são as métricas que importam.

## 3. Treinar e comparar modelos

Começo pela **regressão logística** como baseline e comparo com **Random Forest** (`class_weight="balanced"`) e **XGBoost** (`scale_pos_weight`, que faz o mesmo papel de dar mais peso à fraude).

Também treino uma regressão logística com `class_weight="balanced"` para isolar o efeito de dar peso à classe rara.

In [ ]:
resultados = {}   # nome -> métricas da classe de fraude
probas = {}       # nome -> probabilidade prevista de fraude no teste


def metricas(y_true, proba, limiar=0.5):
    y_true = np.asarray(y_true)
    pred = (proba >= limiar).astype(int)
    return {
        "limiar": limiar,
        "precisao": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred),
        "f1": f1_score(y_true, pred),
        "roc_auc": roc_auc_score(y_true, proba),
        "pr_auc": average_precision_score(y_true, proba),
        "fraudes_pegas": int(((pred == 1) & (y_true == 1)).sum()),
        "falsos_alarmes": int(((pred == 1) & (y_true == 0)).sum()),
    }


def registrar(nome, proba, limiar=0.5):
    resultados[nome] = metricas(y_test, proba, limiar)
    probas[nome] = proba


def tabela_comparacao():
    cols = ["limiar", "precisao", "recall", "f1", "roc_auc", "pr_auc", "fraudes_pegas", "falsos_alarmes"]
    return pd.DataFrame(resultados).T[cols].sort_values("f1", ascending=False).round(4)


spw = (y_train == 0).sum() / (y_train == 1).sum()   # peso da fraude para o XGBoost
print(f"scale_pos_weight = {spw:.1f}")


def novo_xgb(**kwargs):
    return XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="aucpr", tree_method="hist",
        n_jobs=-1, random_state=RANDOM_STATE, **kwargs,
    )

In [ ]:
N_LR   = "Regressão Logística (baseline)"
N_LRB  = "Regressão Logística (balanced)"
N_RF   = "Random Forest (balanced)"
N_XGB  = "XGBoost (scale_pos_weight)"

lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE).fit(X_train_s, y_train)
registrar(N_LR, lr.predict_proba(X_test_s)[:, 1])

lr_bal = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE).fit(X_train_s, y_train)
registrar(N_LRB, lr_bal.predict_proba(X_test_s)[:, 1])

rf = RandomForestClassifier(
    n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE
).fit(X_train_s, y_train)
registrar(N_RF, rf.predict_proba(X_test_s)[:, 1])

xgb = novo_xgb(scale_pos_weight=spw).fit(X_train_s, y_train)
registrar(N_XGB, xgb.predict_proba(X_test_s)[:, 1])

tabela_comparacao()

## 4. Avaliar

Relatório de classificação (limiar 0,5), olhando a linha **fraude (1)**:

In [ ]:
for nome in [N_LR, N_LRB, N_RF, N_XGB]:
    pred = (probas[nome] >= 0.5).astype(int)
    print("=" * 62)
    print(nome)
    print(classification_report(y_test, pred, target_names=["normal (0)", "fraude (1)"], digits=4))

### Curva ROC e curva de precisão × recall

A ROC costuma parecer ótima mesmo em modelos medianos, porque os verdadeiros negativos (milhares) puxam a taxa de falsos positivos para perto de zero. A **curva de precisão × recall** não usa os negativos e mostra a dificuldade real. A linha tracejada é um modelo aleatório (na PR ele fica na proporção de fraudes).

In [ ]:
modelos_curvas = [N_LR, N_LRB, N_RF, N_XGB]

fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))

for nome in modelos_curvas:
    fpr, tpr, _ = roc_curve(y_test, probas[nome])
    ax[0].plot(fpr, tpr, label=f"{nome} (AUC={resultados[nome]['roc_auc']:.3f})")

    prec, rec, _ = precision_recall_curve(y_test, probas[nome])
    ax[1].plot(rec, prec, label=f"{nome} (AP={resultados[nome]['pr_auc']:.3f})")

ax[0].plot([0, 1], [0, 1], "k--", label="aleatório")
ax[0].set_title("Curva ROC")
ax[0].set_xlabel("Taxa de falsos positivos")
ax[0].set_ylabel("Taxa de verdadeiros positivos (recall)")
ax[0].legend(loc="lower right", fontsize=8)

ax[1].axhline(y_test.mean(), color="k", ls="--", label=f"aleatório ({y_test.mean():.4f})")
ax[1].set_title("Curva de precisão × recall (classe fraude)")
ax[1].set_xlabel("Recall")
ax[1].set_ylabel("Precisão")
ax[1].legend(loc="lower left", fontsize=8)

plt.tight_layout()
plt.show()

## 5. Ajustar: Pipeline, GridSearchCV e limiar de decisão

**Pipeline:** junta o `StandardScaler` e o modelo. Dentro do `GridSearchCV`, o scaler é reajustado em cada dobra só com os dados de treino daquela dobra, então não há vazamento.

**GridSearchCV escolhendo pelo recall:** uso `refit="recall"`, mas guardo também precisão, F1 e PR-AUC de cada combinação. Escolher só pelo recall tem uma armadilha: marcar *tudo* como fraude daria recall de 100%. Por isso olho a tabela de cada combinação e não apenas o vencedor.

Faço isso para dois modelos: regressão logística (rápida) e XGBoost.

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "recall": "recall",
    "precision": "precision",
    "f1": "f1",
    "pr_auc": "average_precision",
}
cols_cv = ["params", "mean_test_recall", "mean_test_precision", "mean_test_f1", "mean_test_pr_auc"]

# --- Regressão logística em Pipeline
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
])
grid_lr = GridSearchCV(
    pipe_lr, {"clf__C": [0.01, 0.1, 1, 10]},
    scoring=scoring, refit="recall", cv=cv, n_jobs=-1,
)
grid_lr.fit(X_train, y_train)

print("Melhor (LR):", grid_lr.best_params_)
pd.DataFrame(grid_lr.cv_results_)[cols_cv].round(4)

In [ ]:
# --- XGBoost em Pipeline
pipe_xgb = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", novo_xgb(scale_pos_weight=spw)),
])
param_xgb = {
    "clf__max_depth": [3, 5],
    "clf__n_estimators": [200, 400],
}
grid_xgb = GridSearchCV(
    pipe_xgb, param_xgb,
    scoring=scoring, refit="recall", cv=cv, n_jobs=1,
)
grid_xgb.fit(X_train, y_train)

print("Melhor (XGBoost):", grid_xgb.best_params_)
pd.DataFrame(grid_xgb.cv_results_)[cols_cv].round(4)

### Limiar de decisão: 0,5 → 0,3

O modelo devolve uma **probabilidade** de fraude; o limiar transforma isso em "fraude / não fraude". Baixar de 0,5 para 0,3 marca mais transações como fraude: **o recall sobe e a precisão cai** (mais falsos alarmes). Abaixo comparo os dois limiares nos dois pipelines.

In [ ]:
LIMIAR_PADRAO = 0.5
LIMIAR_AJUSTADO = 0.3

best_lr = grid_lr.best_estimator_
best_xgb = grid_xgb.best_estimator_

N_LR_PIPE  = "LR Pipeline + GridSearch"
N_XGB_PIPE = "XGBoost Pipeline + GridSearch"

for nome, modelo in [(N_LR_PIPE, best_lr), (N_XGB_PIPE, best_xgb)]:
    proba = modelo.predict_proba(X_test)[:, 1]      # o Pipeline padroniza sozinho
    for limiar in [LIMIAR_PADRAO, LIMIAR_AJUSTADO]:
        registrar(f"{nome} (limiar {limiar})", proba, limiar)

tabela_comparacao()

In [ ]:
def varrer_limiar(y_true, proba, limiares=(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9)):
    y_true = np.asarray(y_true)
    linhas = []
    for t in limiares:
        pred = (proba >= t).astype(int)
        linhas.append({
            "limiar": t,
            "precisao": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred),
            "f1": f1_score(y_true, pred),
            "fraudes_pegas": int(((pred == 1) & (y_true == 1)).sum()),
            "falsos_alarmes": int(((pred == 1) & (y_true == 0)).sum()),
        })
    return pd.DataFrame(linhas).set_index("limiar").round(4)


proba_xgb_pipe = probas[f"{N_XGB_PIPE} (limiar {LIMIAR_PADRAO})"]
varrer_limiar(y_test, proba_xgb_pipe)

In [ ]:
prec, rec, _ = precision_recall_curve(y_test, proba_xgb_pipe)

plt.figure(figsize=(7, 5.5))
plt.plot(rec, prec, color="#4C78A8", label=N_XGB_PIPE)
for limiar, cor in [(LIMIAR_PADRAO, "#E45756"), (LIMIAR_AJUSTADO, "#54A24B")]:
    m = resultados[f"{N_XGB_PIPE} (limiar {limiar})"]
    plt.scatter(m["recall"], m["precisao"], s=90, color=cor, zorder=3, label=f"limiar {limiar}")
plt.xlabel("Recall")
plt.ylabel("Precisão")
plt.title("Onde cada limiar cai na curva precisão × recall")
plt.legend()
plt.tight_layout()
plt.show()

**Escolha do limiar.** Vou usar **0,3**, como no roteiro: ele pega mais fraudes que o 0,5 em troca de alguns falsos alarmes a mais. Em um banco real, a escolha depende do custo: um falso negativo é uma fraude que passa (prejuízo direto), e um falso positivo é um cliente com cartão bloqueado à toa. Se bloquear à toa for caro, o limiar sobe; se deixar fraude passar for mais caro, ele desce. Confira a tabela de varredura acima antes de aceitar o 0,3.

## 6. Experimento extra: undersampling × oversampling × pesos de classe

Existem três formas comuns de lidar com o desbalanceamento. Em vez de escolher uma por opinião, eu testo:

- **sem tratamento**: modelo padrão
- **pesos de classe**: `class_weight="balanced"` / `scale_pos_weight`
- **undersampling**: reduzo as transações normais do treino até igualar o número de fraudes
- **oversampling**: repito fraudes do treino (com reposição) até igualar o número de normais

Regra importante: o reamostragem acontece **só no treino, depois do split**. O teste continua intacto e com a proporção real. Reamostrar antes de separar contaminaria o teste.

In [ ]:
treino = X_train_s.copy()
treino["Class"] = y_train.values

fraudes_tr = treino[treino["Class"] == 1]
normais_tr = treino[treino["Class"] == 0]

under = pd.concat([normais_tr.sample(len(fraudes_tr), random_state=RANDOM_STATE), fraudes_tr])
under = under.sample(frac=1, random_state=RANDOM_STATE)

over = pd.concat([normais_tr, fraudes_tr.sample(len(normais_tr), replace=True, random_state=RANDOM_STATE)])
over = over.sample(frac=1, random_state=RANDOM_STATE)

conjuntos = {
    "sem tratamento":  (X_train_s, y_train, False),
    "pesos de classe": (X_train_s, y_train, True),
    "undersampling":   (under.drop(columns="Class"), under["Class"], False),
    "oversampling":    (over.drop(columns="Class"), over["Class"], False),
}
print({k: (len(v[0]), int(v[1].sum())) for k, v in conjuntos.items()})

In [ ]:
linhas = []
for estrategia, (Xt, yt, usar_peso) in conjuntos.items():
    for modelo in ["Regressão Logística", "XGBoost"]:
        if modelo == "Regressão Logística":
            m = LogisticRegression(
                max_iter=1000, class_weight="balanced" if usar_peso else None,
                random_state=RANDOM_STATE,
            )
        else:
            m = novo_xgb(scale_pos_weight=spw if usar_peso else 1)
        m.fit(Xt, yt)
        proba = m.predict_proba(X_test_s)[:, 1]      # sempre no teste original
        linhas.append({"modelo": modelo, "estrategia": estrategia, **metricas(y_test, proba)})

tab_amostragem = (
    pd.DataFrame(linhas)
    .set_index(["modelo", "estrategia"])
    [["precisao", "recall", "f1", "pr_auc", "fraudes_pegas", "falsos_alarmes"]]
    .round(4)
)
tab_amostragem

Repare no padrão: under/oversampling costumam **subir o recall e derrubar a precisão** (muitos falsos alarmes), e o efeito muda de modelo para modelo. É por isso que a recomendação é testar, e comparar pelas métricas da classe de fraude, não pela acurácia.

## 7. Explicar o modelo

### Importância das variáveis

`feature_importances_` mostra quais variáveis o modelo mais usa **no geral**. Não diz em que direção (se o valor alto empurra para fraude ou para normal). Para isso serve o SHAP.

In [ ]:
imp_rf = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)
imp_xgb = pd.Series(best_xgb.named_steps["clf"].feature_importances_, index=X.columns).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
imp_rf[::-1].plot.barh(ax=ax[0], color="#4C78A8")
ax[0].set_title("Random Forest: top 15 variáveis")
imp_xgb[::-1].plot.barh(ax=ax[1], color="#72B7B2")
ax[1].set_title("XGBoost (pipeline): top 15 variáveis")
plt.tight_layout()
plt.show()

### SHAP

O SHAP atribui a cada variável quanto ela **empurrou** a previsão de cada transação para cima (rumo a fraude) ou para baixo (rumo a normal). Explico o XGBoost do pipeline. Os valores estão em log-odds.

Para o gráfico global uso uma amostra do teste com 2.000 transações normais **mais todas as fraudes do teste**. Assim as fraudes aparecem o bastante para o gráfico mostrar o que as caracteriza (a amostra, portanto, não representa a proporção real).

In [ ]:
modelo_xgb = best_xgb.named_steps["clf"]
X_test_final = pd.DataFrame(
    best_xgb.named_steps["scaler"].transform(X_test), columns=X.columns, index=X_test.index
)

explainer = shap.TreeExplainer(modelo_xgb)

amostra = pd.concat([
    X_test_final[y_test == 0].sample(2000, random_state=RANDOM_STATE),
    X_test_final[y_test == 1],
])
sv = explainer(amostra)

In [ ]:
shap.plots.beeswarm(sv, max_display=15, show=False)
plt.title("SHAP: impacto de cada variável na previsão de fraude")
plt.tight_layout()
plt.show()

In [ ]:
shap.plots.bar(sv, max_display=15, show=False)
plt.title("SHAP: impacto médio absoluto por variável")
plt.tight_layout()
plt.show()

### Explicando decisões individuais

Duas transações do teste, usando o limiar escolhido (0,3):

- **uma fraude que o modelo pegou** (verdadeiro positivo): o que pesou para marcá-la
- **uma fraude que o modelo deixou passar** (falso negativo): o que a fez parecer normal

In [ ]:
proba_final = probas[f"{N_XGB_PIPE} (limiar {LIMIAR_AJUSTADO})"]
pred_final = proba_final >= LIMIAR_AJUSTADO
y_arr = y_test.values

idx_tp = np.where((y_arr == 1) & pred_final)[0]
idx_fn = np.where((y_arr == 1) & ~pred_final)[0]
print(f"Fraudes pegas: {len(idx_tp)} | fraudes perdidas: {len(idx_fn)}")

if len(idx_tp):
    i = idx_tp[0]
    sv_tp = explainer(X_test_final.iloc[[i]])
    shap.plots.waterfall(sv_tp[0], max_display=12, show=False)
    plt.title(f"Fraude PEGA (prob. = {proba_final[i]:.3f})")
    plt.tight_layout()
    plt.show()

In [ ]:
if len(idx_fn):
    i = idx_fn[np.argmax(proba_final[idx_fn])]     # a fraude perdida mais "perto" de ser pega
    sv_fn = explainer(X_test_final.iloc[[i]])
    shap.plots.waterfall(sv_fn[0], max_display=12, show=False)
    plt.title(f"Fraude PERDIDA (prob. = {proba_final[i]:.3f})")
    plt.tight_layout()
    plt.show()
else:
    print("Nenhuma fraude perdida no limiar escolhido.")

## 8. Resultado final

Modelo final: **XGBoost em Pipeline, ajustado com GridSearchCV, limiar 0,3.**

In [ ]:
nome_final = f"{N_XGB_PIPE} (limiar {LIMIAR_AJUSTADO})"
pred_final_int = pred_final.astype(int)

print(nome_final)
print(classification_report(y_test, pred_final_int, target_names=["normal (0)", "fraude (1)"], digits=4))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, pred_final_int), display_labels=["normal", "fraude"]
).plot(cmap="Blues", values_format="d")
plt.title("Matriz de confusão: modelo final (teste)")
plt.show()

### Tabelas prontas para o README

A célula abaixo imprime as tabelas em Markdown para você copiar para o `README.md`.

In [ ]:
tabela_final = tabela_comparacao().astype({"fraudes_pegas": int, "falsos_alarmes": int})

print("### Comparação de modelos (classe de fraude, teste)\n")
print(tabela_final.to_markdown())

print("\n\n### Estratégias de balanceamento (classe de fraude, teste, limiar 0,5)\n")
print(tab_amostragem.to_markdown())

## Limitações

- Os dados são de transações europeias de 2013; padrões de fraude mudam com o tempo, então o modelo precisaria ser retreinado e monitorado.
- A divisão treino/teste é aleatória (estratificada). Em produção o correto seria dividir **por tempo** (treinar no passado, testar no futuro), porque `Time` indica que as transações são sequenciais.
- Com ~98 fraudes no teste, as métricas variam bastante entre sementes e divisões. Vale repetir com validação cruzada para ver a dispersão.
- As variáveis `V1` a `V28` são anonimizadas, então o SHAP diz **quais** pesaram, mas não dá para traduzir isso em uma regra de negócio legível.